# 07 · La campaña: "El sur tiene espacio"
**Torre del Caribe** · Fase 8 (Mercadotecnia digital) · Autor: **Brandon Uriel García Sánchez** · UNRC, LCDN 5° semestre 2026-2

| | |
|---|---|
| **Qué hace** | Convierte los datos en una campaña: qué molesta y qué enamora en las reseñas, a quién se le habla, con qué palabras, en qué canal, cuándo y cómo se mide. |
| **Decisiones de Brandon** | Dos personas (la que vuelve al sur; la que baja del norte) · nombre "El sur tiene espacio" · tono que combina calma, orgullo cultural y aventura (`docs/decisiones/21-campana.md`). |
| **Código** | `backend/torre/campana/texto.py` (minería) y `marca.py` (campaña). |
| **Ecuaciones** | `docs/metodologia/ECUACIONES.md` §6.0. |

In [1]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "backend"))
import pandas as pd
from torre.campana import texto as tx, marca as mk
pd.set_option("display.max_colwidth", 120)

## 1. Temas de las reseñas y su riesgo
Léxico a la vista (`tx.ASPECTOS`). Riesgo relativo $RR_a=\dfrac{P(\text{mala}\mid a)}{P(\text{mala})}$: más de 1, el tema
aleja; menos de 1, protege.

In [2]:
r = tx.marcar_aspectos(tx.leer())
print(f"{len(r):,} reseñas · malas {r.mala.mean():.2%} · de 5 estrellas {r.cinco.mean():.2%}")
a = tx.aspectos(r)
a[a.pueblo == "Quintana Roo (3 pueblos)"].sort_values("riesgo_relativo", ascending=False).round(2)

85,987 reseñas · malas 4.18% · de 5 estrellas 69.87%


,pueblo,aspecto,resenas,pct_menciona,pct_malas,pct_malas_pueblo,riesgo_relativo
30,Quintana Roo (3 pueblos),ruido,2038,2.37,11.53,4.18,2.76
29,Quintana Roo (3 pueblos),limpieza,9520,11.07,7.92,4.18,1.89
28,Quintana Roo (3 pueblos),precio,21999,25.58,6.96,4.18,1.66
27,Quintana Roo (3 pueblos),multitudes,15434,17.95,6.35,4.18,1.52
35,Quintana Roo (3 pueblos),servicio,38041,44.24,4.95,4.18,1.18
34,Quintana Roo (3 pueblos),comida,34696,40.35,3.40,4.18,0.81
32,Quintana Roo (3 pueblos),naturaleza,11971,13.92,3.06,4.18,0.73
33,Quintana Roo (3 pueblos),cultura,15721,18.28,1.99,4.18,0.48
31,Quintana Roo (3 pueblos),calma,10572,12.29,1.72,4.18,0.41


## 2. Reglas de asociación (Apriori)
Transacciones = temas mencionados + la calificación. soporte $=\#\{A\cup B\}/N$, confianza $=sop(A\cup B)/sop(A)$,
lift $=conf/sop(B)$.

In [3]:
reg = tx.reglas_asociacion(r)
display(reg[reg.entonces.str.contains("mala")].head(8).round(4))
reg[reg.entonces == "reseña de 5"].head(6).round(4)

,si,entonces,soporte,confianza,lift
447,ruido + servicio,reseña mala (1–2),0.0017,0.1179,2.8194
40,ruido,reseña mala (1–2),0.0027,0.1153,2.7565
137,limpieza + multitudes,reseña mala (1–2),0.0027,0.1084,2.5924
256,limpieza + precio,reseña mala (1–2),0.0039,0.1062,2.5395
340,precio + servicio,reseña mala (1–2),0.0094,0.0851,2.0337
97,multitudes + precio,reseña mala (1–2),0.0047,0.0837,2.0007
221,multitudes + servicio,reseña mala (1–2),0.0059,0.0824,1.9708
35,limpieza,reseña mala (1–2),0.0088,0.0792,1.8933


,si,entonces,soporte,confianza,lift
516,comida + servicio,reseña de 5,0.2010,0.7813,1.1182
464,calma + naturaleza,reseña de 5,0.0172,0.7663,1.0967
53,comida,reseña de 5,0.3053,0.7567,1.0829
480,calma + comida,reseña de 5,0.0361,0.7553,1.0809
486,calma + servicio,reseña de 5,0.0456,0.7504,1.0740
57,servicio,reseña de 5,0.3280,0.7414,1.0610


## 3. El lenguaje del turista (log-odds con prior de Dirichlet informativo)

In [4]:
p = tx.palabras_que_distinguen(r)
p.groupby("lado").palabra.apply(lambda x: ", ".join(x))

lado
reseñas de 1–2 estrellas    habitación, dijeron, peor, horrible, dinero, terrible, caro, hotel, dijo, decepcionante, nadie, decepción, pedimos, ...
reseñas de 5 estrellas      excelente, increíble, playa, tulum, comida, isla, ruinas, hermoso, deliciosa, amable, encantó, ambiente, gracias, du...
Name: palabra, dtype: object

## 4. Las dos viajeras ideales
Cada rasgo dice si es **dato**, **derivado**, **supuesto** o **hueco**.

In [5]:
c = json.loads((RAIZ / "datos" / "gold" / "campana.json").read_text(encoding="utf-8"))
for persona in c["personas"]:
    print("\n" + persona["nombre"].upper(), "—", persona["quien"])
    display(pd.DataFrame(persona["atributos"], columns=["rasgo", "valor", "tipo", "fuente"]))


LA QUE VUELVE AL SUR — Viajera mexicana que ya conoce o tiene cerca el sur de Quintana Roo y busca un viaje con historia, comida y calma.


,rasgo,valor,tipo,fuente
0,Origen,Nacional: 94.8 % de los visitantes de Oxtankah y 63.3 % de los de la Ruta en 2025,dato,INAH 2025
1,Estado de origen,No se publica de qué estado vienen los visitantes nacionales,hueco,—
2,Cuándo viaja,"Diciembre a abril (temporada alta del sur); la campaña la invita en mayo–junio y octubre–noviembre, con espacio",derivado,"Forma del año, Fase 5"
3,Qué valora,"Calma y cultura: una reseña que habla de calma es mala 0.41 veces lo normal; de cultura, 0.48",derivado,"Rest-Mex 2025, texto.py"
4,Qué la aleja,"Ruido (2.8×), suciedad (1.9×), precio (1.7×) y multitudes (1.5×) en reseñas malas",derivado,Rest-Mex 2025
5,Cómo se informa,Mensajería (90.6 %) y redes sociales (80.4 %) en el teléfono,dato,"ENDUTIH 2025, pág. 17"
6,Edad e ingreso,No hay dato oficial por destino: no se fija,hueco,—



LA QUE BAJA DEL NORTE — Viajera de Estados Unidos o Canadá que ya está en Cancún o la Riviera Maya y puede bajar al sur en Tren Maya o por carretera.


,rasgo,valor,tipo,fuente
0,Origen,"Estados Unidos 56.3 % y Canadá 17.1 % de los 9,408,423 extranjeros que llegaron a Cancún en 2025",dato,UPM vía DataTur
1,Sexo,Mujeres 53.4 %,dato,UPM vía DataTur
2,Por qué no llega sola al sur,El aeropuerto de Chetumal recibió 250 extranjeros en 2025,dato,UPM vía DataTur
3,Cuándo está,"Más llegadas en marzo, menos en septiembre; Cancún en temporada alta de noviembre a abril",dato,UPM; planeador Fase 5
4,Qué la aleja del norte,"Multitudes, precio y sargazo (destinos excluidos por sargazo 2026)",derivado,Rest-Mex; REGIONES.md
5,Cómo se informa,Busca en Google y usa redes desde el destino,supuesto,Benchmarks de viajes (D13); sin encuesta propia
6,Idioma,Inglés (y español),derivado,País de origen


## 5. Marca, mensajes, medios, calendario y KPI

In [6]:
print(json.dumps(c["marca"], ensure_ascii=False, indent=1))
print("Revisión de largos de anuncios:", mk.revisar_largos(c["mensajes"]) or "todo dentro de los límites")
display(pd.DataFrame(c["mensajes"])[["persona", "canal", "idioma", "respaldo"]])
display(pd.DataFrame(c["medios"]))
display(pd.DataFrame(c["calendario"]))
pd.DataFrame(c["kpis"])

{
 "nombre": "El sur tiene espacio",
 "lema": "Pirámides, bahía y sabor del sur, con espacio para disfrutarlos.",
 "nombre_en": "The south has room",
 "lema_en": "Mayan pyramids, a quiet bay and southern food, with room to enjoy them.",
 "personalidad": [
  "Cercana y tranquila: habla de tú, frases cortas, promete calma.",
  "Orgullosa de lo maya y de lo mexicano: Kohunlich, Dzibanché, Ichkabal, Oxtankah.",
  "Con espíritu de aventura: selva, caminos y ruinas para recorrer sin prisa."
 ],
 "propuesta_valor": "El Caribe mexicano que todavía tiene espacio: zonas mayas con 15.5 veces menos visitantes que Tulum, una bahía tranquila y la comida del sur.",
 "posicionamiento": "Para quien quiere el Caribe mexicano sin multitudes, el sur de Quintana Roo es la opción cultural y tranquila; a diferencia de Tulum o Cancún, aquí hay espacio, y la campaña lo cuida: nunca anuncia un lugar lleno.",
 "visual": "Sistema \"Sur mexicano\" (decisión 07): rosa, amarillo, turquesa y añil en bloques; Bricolag

,persona,canal,idioma,respaldo
0,vuelve,Google (búsqueda),es,"INAH 2025: Tulum 1,031,443 visitantes; Ruta 66,628 (15.5 veces menos)"
1,vuelve,Facebook / Instagram,es,"Oxtankah recibió 11,017 visitantes en 2025 (INAH), 94.8 % mexicanos"
2,vuelve,WhatsApp (orgánico),es,"90.6 % usa mensajería en el teléfono (ENDUTIH 2025, pág. 17); sin costo"
3,baja,Google (búsqueda),en,"INAH 2025: Tulum 1,031,443 visitantes; Ruta 66,628 (15.5 veces menos); Cancún en temporada alta nov–abr (planeador)"
4,baja,Facebook / Instagram,en,"Tren Maya con estación en Chetumal (SITUR-Q, descensos por estación); solo se muestra si la Torre marca norte lleno ..."


,medio,pct,por_que
0,Facebook / Instagram (pagado),70,"Rinde 6.61 visitantes por cada $1,000 (clic de $0.51 USD, WordStream 2025) y 80.4 % usa redes (ENDUTIH 2025). Tope d..."
1,"Google, búsqueda (pagado)",30,Llega a quien ya busca viajar (conversión medida de 5.75 % en Travel). Es el canal de la persona que baja del norte.
2,WhatsApp y compartir (orgánico),0,90.6 % usa mensajería: el botón de compartir de la página no cuesta y viaja entre familia y amigos.
3,La página web (destino de todos los anuncios),0,"Planeador por lugar y mes en 9 idiomas, sin internet externo; recomienda otro lugar del sur si el elegido está lleno."


,mes,pesos,lugares,personas
0,2026-10,32586,"[Bahía Calderitas–Oxtankah, Chetumal, Ruta arqueológica del sur]",[vuelve]
1,2026-11,28012,"[Bahía Calderitas–Oxtankah, Chetumal, Ruta arqueológica del sur]","[vuelve, baja]"
2,2026-12,0,[],[]
3,2027-01,5650,[Chetumal],"[vuelve, baja]"
4,2027-02,25339,"[Bahía Calderitas–Oxtankah, Chetumal, Ruta arqueológica del sur]","[vuelve, baja]"
5,2027-03,12073,"[Bahía Calderitas–Oxtankah, Chetumal]","[vuelve, baja]"
6,2027-04,14496,"[Chetumal, Ruta arqueológica del sur]","[vuelve, baja]"
7,2027-05,33359,"[Bahía Calderitas–Oxtankah, Chetumal, Ruta arqueológica del sur]",[vuelve]
8,2027-06,35985,"[Bahía Calderitas–Oxtankah, Chetumal, Ruta arqueológica del sur]",[vuelve]


,kpi,formula,meta,fuente,frecuencia
0,Alcance,Personas únicas que vieron un anuncio,Se fija con el primer mes real (no hay dato previo),Plataformas,Semanal
1,Interacción (CTR),Clics ÷ impresiones,Google ≥ 8.73 %; Facebook ≥ 2.76 % (promedios de Travel),Plataformas; WordStream 2025,Semanal
2,Conversión,Planes de viaje o contactos ÷ clics,≥ 5.75 % (Google Travel),Página web,Semanal
3,Costo por visitante,Pesos gastados ÷ conversiones,≤ $196 (modelo de la Fase 6),Plataformas + página,Mensual
4,Afluencia,Visitantes reales − escenario probable del pronóstico,"+957 en 9 meses, sin pasar la capacidad probada","INAH, SITUR-Q (Belice)",Mensual
5,Presupuesto,Pesos gastados ÷ pesos planeados,100 % (lo pausado se gasta después),Torre en vivo,Semanal
6,Económico,Ocupación hotelera de Chetumal,Hueco: SITUR-Q no publica ocupación del sur desde 2025,—,—
7,Social,"Visitantes por cada 1,000 habitantes (Radar)","El lugar sigue ""tranquilo"" o ""concurrido"", nunca ""saturado""","Radar, Fase 4",Mensual
8,Ambiental,Semanas con anuncio en temporada alta o con mal clima,0 (la Torre las pausa),"Torre en vivo, Fase 7",Semanal
9,Capacidad,(Esperados + campaña) ÷ capacidad probada,"≤ 40 % en los meses con anuncio (Pareto, Fase 6)",Pronóstico + presupuesto,Mensual


## 6. Qué se concluye
1. **Lo que hunde una reseña es lo que sobra en los destinos llenos:** ruido (2.76×), suciedad (1.89×), precio (1.66×) y
   multitudes (1.52×). Lo que la protege es lo que el sur ofrece: calma (0.41×) y cultura (0.48×).
2. **Dos viajeras, dos caminos:** la nacional ya es el 95 % de la Bahía; la extranjera está en Cancún (9.4 millones en
   2025) y casi no llega al sur por avión (250 en Chetumal). Por eso hay un mensaje para volver y otro para bajar.
3. **Ningún anuncio promete lo que no está medido:** sin precios ni horas de viaje; "con espacio" se apoya en 15.5 veces
   menos visitantes que Tulum.
4. **Limitaciones:** Rest-Mex no cubre los cinco lugares; muchas reseñas son traducciones; no hay dato del estado de
   origen del visitante nacional ni de edad o ingreso; los canales y conversiones son promedios de EE. UU.